In [9]:

import sys
import os
import pandas as pd
import folium
from folium import plugins
import branca.colormap as cm
import numpy as np  # Add missing numpy import

# Add the paths to import from other notebooks
sys.path.append('/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/Wind/Digital_Twin/Response_hurricane')

# Import SPARQL functions
from SPARQLWrapper import SPARQLWrapper, JSON, GET

def setup_sparql_connection():
    """Setup SPARQL connection and helper functions"""
    fuseki_endpoint = "http://localhost:3030/Wind/sparql"
    sparql = SPARQLWrapper(fuseki_endpoint)
    
    def execute_select_query(query):
        sparql.setQuery(query)
        sparql.setReturnFormat(JSON)
        sparql.setMethod(GET)
        try:
            results = sparql.query().convert()
            return results
        except Exception as e:
            print(f"Error executing query: {e}")
            return None
    
    def results_to_dataframe(results):
        if not results or 'results' not in results:
            return pd.DataFrame()
        
        bindings = results['results']['bindings']
        if not bindings:
            return pd.DataFrame()
        
        columns = list(bindings[0].keys())
        data = []
        for binding in bindings:
            row = {}
            for col in columns:
                if col in binding:
                    row[col] = binding[col]['value']
                else:
                    row[col] = None
            data.append(row)
        
        return pd.DataFrame(data)
    
    return execute_select_query, results_to_dataframe

def load_turbine_data():
    """Load turbine data from SPARQL endpoint"""
    execute_select_query, results_to_dataframe = setup_sparql_connection()
    
    all_turbines_query = """
    PREFIX turbine: <http://www.cee.umd.edu/Energy/Turbine#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    
    SELECT ?turbine ?property ?value
    WHERE {
        ?turbine rdf:type turbine: .
        ?turbine ?property ?value .
        FILTER(STRSTARTS(STR(?property), "http://www.cee.umd.edu/Energy/Turbine#"))
    }
    ORDER BY ?turbine ?property
    """
    
    all_results = execute_select_query(all_turbines_query)
    all_data_df = results_to_dataframe(all_results)
    all_data_df['property_name'] = all_data_df['property'].str.replace('http://www.cee.umd.edu/Energy/Turbine#', '')
    
    turbines_df = all_data_df.pivot_table(
        index='turbine', 
        columns='property_name', 
        values='value', 
        aggfunc='first'
    ).reset_index()
    
    return turbines_df

def load_coordinate_data():
    """Load turbine and wind farm coordinates"""
    turbines_coord = "/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/Wind/Digital_Twin/Coordinates/MD turbine coordinates.txt"
    windfarm_coord = "/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/Wind/Digital_Twin/Coordinates/MD wind farm coordinates.txt"
    
    # Load turbine coordinates
    try:
        turbine_locations = pd.read_csv(turbines_coord, header=None, names=['longitude', 'latitude'])
        print(f"Loaded {len(turbine_locations)} turbine locations")
    except Exception as e:
        print(f"Error loading turbine coordinates: {e}")
        return None, None
    
    # Load wind farm polygon
    try:
        with open(windfarm_coord, 'r') as f:
            content = f.read().strip()
        
        if content.startswith('POLYGON'):
            coords_str = content.replace('POLYGON ((', '').replace('))', '')
            coord_pairs = coords_str.split(', ')
            
            windfarm_polygon = []
            for pair in coord_pairs:
                lon, lat = pair.split(' ')
                windfarm_polygon.append([float(lat), float(lon)])
            
            print(f"Loaded wind farm polygon with {len(windfarm_polygon)} points")
        else:
            print("Wind farm coordinate file format not recognized")
            windfarm_polygon = None
    except Exception as e:
        print(f"Error loading wind farm coordinates: {e}")
        windfarm_polygon = None
    
    return turbine_locations, windfarm_polygon

def create_merged_hurricane_turbine_map(hurricane_data, turbines_df, turbine_locations, windfarm_polygon, 
                                       closest_approach_idx, title='Hurricane Sandy - Wind Farm Impact'):
    """Create merged map showing hurricane at closest approach with detailed turbine information"""
    
    # Get closest approach data
    closest_data = hurricane_data.iloc[closest_approach_idx]
    hurricane_lat = closest_data['latitude']
    hurricane_lon = closest_data['longitude']
    hurricane_wind = closest_data['max_wind']  # knots
    datetime_str = closest_data['datetime'].strftime('%Y-%m-%d %H:%M')
    distance_to_farm = closest_data['distance_km']
    
    # Calculate wind farm center
    if windfarm_polygon:
        lats = [coord[0] for coord in windfarm_polygon]
        lons = [coord[1] for coord in windfarm_polygon]
        farm_center_lat = sum(lats) / len(lats)
        farm_center_lon = sum(lons) / len(lons)
    else:
        farm_center_lat = 38.27
        farm_center_lon = -74.68
    
    # Create map centered between hurricane and wind farm
    center_lat = (hurricane_lat + farm_center_lat) / 2
    center_lon = (hurricane_lon + farm_center_lon) / 2
    
    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=8,
        tiles=None
    )
    
    # Add map layers
    folium.TileLayer(
        tiles='OpenStreetMap',
        name='OpenStreetMap',
        overlay=False,
        control=True
    ).add_to(m)
    
    folium.TileLayer(
        tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
        attr='Esri',
        name='Satellite',
        overlay=False,
        control=True
    ).add_to(m)
    
    # Add hurricane center
    folium.Marker(
        location=[hurricane_lat, hurricane_lon],
        popup=folium.Popup(f'''
        <b style="font-size: 13px;">Hurricane Center - Closest Approach</b><br>
        <span style="font-size: 13px;">Time: {datetime_str}</span><br>
        <span style="font-size: 13px;">Max Wind: {hurricane_wind * 0.514:.1f} m/s</span><br>
        <span style="font-size: 13px;">Distance to Wind Farm: {distance_to_farm:.1f} km</span>
        ''', max_width=250),
        tooltip=f'Hurricane Center: {hurricane_wind * 0.514:.1f} m/s',
        icon=folium.Icon(color='red', icon='exclamation-triangle', prefix='fa')
    ).add_to(m)
    
    # Define helper functions BEFORE using them
    def holland_wind_profile(r, vmax, rmax, B=1.5):
        """
        Holland wind profile model that matches Hurricane V1 implementation.
        
        Physics:
        - Hurricane eye (r ≈ 0): Very low wind speeds
        - Eyewall (r = rmax): Maximum wind speeds (vmax)  
        - Outer bands (r > rmax): Decreasing wind speeds with distance
        
        Parameters:
        -----------
        r : float or array
            Distance(s) from hurricane center in km
        vmax : float
            Maximum wind speed at the eyewall in knots
        rmax : float
            Radius of maximum wind (eyewall radius) in km
        B : float
            Holland B parameter (shape parameter, typically 1.0-2.5)
            
        Returns:
        --------
        float or array
            Wind speed at distance r (in same units as vmax)
        """
        # Handle single values and arrays consistently
        if np.isscalar(r):
            if r == 0:
                return vmax
            return vmax * (rmax/r)**B * np.exp(1 - (rmax/r)**B)
        
        # For arrays, vectorized calculation
        r = np.asarray(r)
        wind_speeds = np.zeros_like(r, dtype=float)
        
        # Handle center points (r ≈ 0)
        center_mask = r < 0.001
        wind_speeds[center_mask] = vmax
        
        # Handle non-center points
        non_center_mask = ~center_mask
        r_non_center = r[non_center_mask]
        
        if len(r_non_center) > 0:
            wind_speeds[non_center_mask] = vmax * (rmax/r_non_center)**B * np.exp(1 - (rmax/r_non_center)**B)
        
        return wind_speeds
    
    def adjust_wind_speed_to_height(wind_speed, reference_height=10, turbine_height=150, alpha=0.14):
        """Adjust wind speed to turbine height - same as in hurricane script"""
        return wind_speed * (turbine_height / reference_height) ** alpha
    
    # Define hurricane model parameters
    rmax = 50  # Radius of maximum wind in km
    B = 1.5   # Holland B parameter
    
    # Add concentric wind speed circles with proper wind field visualization (matching Res_hurricane)
    wind_radii = [rmax * 2, rmax * 4, rmax * 6, rmax * 8]  # Multiples of RMW like in Res_hurricane
    colors = ['orange', 'yellow', 'lightblue', 'lightgray']
    
    for i, radius_km in enumerate(wind_radii):
        wind_speed_at_radius = holland_wind_profile(radius_km, hurricane_wind, rmax, B)
        
        folium.Circle(
            location=[hurricane_lat, hurricane_lon],
            radius=radius_km * 1000,  # Convert to meters
            color=colors[i],
            weight=2,
            fillColor=colors[i],
            fillOpacity=0.1,
            popup=f'Wind at {radius_km:.0f} km: {wind_speed_at_radius * 0.514:.1f} m/s',
            tooltip=f'{radius_km:.0f} km: {wind_speed_at_radius * 0.514:.1f} m/s'
        ).add_to(m)
    
    # Add eyewall circle (rmax = 50 km) with maximum wind visualization (matching Res_hurricane)
    folium.Circle(
        location=[hurricane_lat, hurricane_lon],
        radius=rmax * 1000,  # Convert km to meters
        color='red',
        weight=3,
        fillColor='red',
        fillOpacity=0.3,
        popup=f'Eyewall - Maximum winds: {hurricane_wind * 0.514:.1f} m/s',
        tooltip=f'Eyewall (radius: {rmax} km) - Max wind: {hurricane_wind * 0.514:.1f} m/s'
    ).add_to(m)
    
    # Calculate actual wind speed at wind farm center (consistent with hurricane script)
    farm_wind_speed_knots = holland_wind_profile(distance_to_farm, hurricane_wind, rmax)  # Use defined rmax
    
    # Apply height adjustment with proper constraint
    hurricane_wind_adjusted = adjust_wind_speed_to_height(hurricane_wind)
    farm_wind_speed_adjusted_knots = adjust_wind_speed_to_height(farm_wind_speed_knots)
    
    # CRITICAL CONSTRAINT: Wind speed at farm NEVER exceeds height-adjusted hurricane max
    farm_wind_speed_adjusted_knots = min(farm_wind_speed_adjusted_knots, hurricane_wind_adjusted)
    
    farm_wind_speed_ms = farm_wind_speed_adjusted_knots * 0.514  # Convert to m/s
    
    # Helper function to get turbine info (matching Res_turbine style)
    def get_turbine_info(turbine_id):
        turbine_row = turbines_df[turbines_df['hasTurbineID'] == f'Turbine{turbine_id}']
        
        if turbine_row.empty:
            return None
        
        turbine_data = turbine_row.iloc[0]
        
        # Format turbine information with safe access (matching Res_turbine exactly)
        cut_out_speed = turbine_data.get('hasCutOutWindSpeed', 'N/A')
        if cut_out_speed != 'N/A':
            try:
                cut_out_speed = f"{float(cut_out_speed):.1f} m/s"
            except:
                cut_out_speed = 'N/A'
        
        pitch_angle = turbine_data.get('hasPitchAngle', 'N/A')
        if pitch_angle != 'N/A':
            try:
                pitch_angle = f"{float(pitch_angle):.1f}°"
            except:
                pitch_angle = 'N/A'
        
        yaw_angle = turbine_data.get('hasYawAngle', 'N/A')
        if yaw_angle != 'N/A':
            try:
                yaw_angle = f"{float(yaw_angle):.1f}°"
            except:
                yaw_angle = 'N/A'
        
        # Get turbine status first
        turbine_status = turbine_data.get('hasTurbineStatus', 'N/A')
        
        # Handle power output - set to 0 if turbine is parked
        power_output = turbine_data.get('hasPowerOutput', 'N/A')
        if turbine_status.lower() == 'parked':
            power_output = '0.0 MW'
        elif power_output != 'N/A':
            try:
                power_output = f"{float(power_output):.1f} MW"
            except:
                power_output = 'N/A'
        
        return {
            'TurbineModel': turbine_data.get('hasTurbineModel', 'N/A'),
            'CutOutWindSpeed': cut_out_speed,
            'PitchAngle': pitch_angle,
            'YawAngle': yaw_angle,
            'PowerOutput': power_output,
            'TurbineStatus': turbine_status
        }
    
    # Add wind farm boundary BEFORE turbines to ensure proper layering
    if windfarm_polygon:
        folium.Polygon(
            locations=windfarm_polygon,
            color='blue',
            weight=3,
            fillColor='lightblue',
            fillOpacity=0.15,
            popup='Wind Farm Boundary',
            tooltip='Wind Farm Area'
        ).add_to(m)
    
    # Add detailed turbine markers with maximum clickability
    if turbine_locations is not None:
        for idx, turbine in turbine_locations.iterrows():
            turbine_id = idx + 1
            turbine_info = get_turbine_info(turbine_id)
            
            if turbine_info is None:
                turbine_info = {
                    'TurbineModel': 'N/A',
                    'CutOutWindSpeed': 'N/A',
                    'PitchAngle': 'N/A',
                    'YawAngle': 'N/A',
                    'PowerOutput': 'N/A',
                    'TurbineStatus': 'Unknown'
                }
            
            # Calculate hurricane impact on this turbine using same methods as hurricane script
            from math import radians, sin, cos, sqrt, atan2
            def haversine_distance(lat1, lon1, lat2, lon2):
                lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
                dlon = lon2 - lon1
                dlat = lat2 - lat1
                a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
                c = 2 * atan2(sqrt(a), sqrt(1-a))
                return 6371 * c  # Earth radius in km
            
            distance_to_turbine = haversine_distance(hurricane_lat, hurricane_lon, 
                                                   turbine['latitude'], turbine['longitude'])
            
            # Calculate wind speed at turbine using same Holland model as hurricane script
            wind_at_turbine_knots = holland_wind_profile(distance_to_turbine, hurricane_wind, 50)
            
            # Apply height adjustment with proper constraint (matching Res_Hurricane)
            wind_at_turbine_adjusted_knots = adjust_wind_speed_to_height(wind_at_turbine_knots)
            
            # CRITICAL CONSTRAINT: Wind speed at turbine NEVER exceeds height-adjusted hurricane max
            wind_at_turbine_adjusted_knots = min(wind_at_turbine_adjusted_knots, hurricane_wind_adjusted)
            
            wind_at_turbine_ms = wind_at_turbine_adjusted_knots * 0.514  # Convert to m/s
            
            # For turbines very close to wind farm center, use the farm wind speed for consistency
            if distance_to_turbine <= 5:  # Within 5 km of center, use farm wind speed
                wind_at_turbine_ms = farm_wind_speed_ms
            
            # Determine turbine color based on status and hurricane impact
            status = turbine_info['TurbineStatus'].lower()
            if status == 'parked':
                status_text = 'Parked'
                status_color = 'red'
            elif wind_at_turbine_ms > 25:  # Above typical cut-out speed
                status_text = 'HURRICANE SHUTDOWN'
                status_color = 'red'
            elif status == 'operational':
                status_text = 'Operational'
                status_color = 'green'
            elif status == 'maintenance':
                status_text = 'Maintenance'
                status_color = 'orange'
            elif status in ['fault', 'error']:
                status_text = turbine_info['TurbineStatus']
                status_color = 'red'
            else:
                status_text = turbine_info['TurbineStatus']
                status_color = 'gray'
            
            # Create detailed popup content
            popup_content = f'''
            <b style="font-size: 13px;">Turbine {turbine_id}</b><br>
            <span style="font-size: 13px;">Model: {turbine_info['TurbineModel']}</span><br>
            <span style="font-size: 13px;">Cut-out Wind Speed: {turbine_info['CutOutWindSpeed']}</span><br>
            <span style="font-size: 13px;">Pitch Angle: {turbine_info['PitchAngle']}</span><br>
            <span style="font-size: 13px;">Yaw Angle: {turbine_info['YawAngle']}</span><br>
            <span style="font-size: 13px;">Power Output: {turbine_info['PowerOutput']}</span><br>
            <span style="font-size: 13px;">Status: <span style="color: {status_color}; font-weight: bold;">{status_text}</span></span><br>
            <hr style="margin: 5px 0;">
            <b style="font-size: 13px;">Hurricane Impact:</b><br>
            <span style="font-size: 13px;">Distance from Hurricane: {distance_to_turbine:.1f} km</span><br>
            <span style="font-size: 13px;">Estimated Wind Speed: {wind_at_turbine_ms:.1f} m/s</span>
            '''
            
            # Use CircleMarker with smaller radius but high z-index for clickability
            folium.CircleMarker(
                location=[turbine['latitude'], turbine['longitude']],
                radius=6,  # Reduced from 12 to 6 for smaller dots
                popup=folium.Popup(popup_content, max_width=300),
                tooltip=f'Turbine {turbine_id} - {status_text}',
                color='white',
                weight=2,  # Reduced border weight
                fillColor=status_color,  # Use status color for better visibility
                fillOpacity=0.9,
                # Ensure high z-index for clickability over wind field
                zIndexOffset=1000
            ).add_to(m)
    
    # Add wind intensity colormap legend (matching Res_hurricane exactly)
    import branca.colormap as cm
    wind_colormap = cm.LinearColormap(
        colors=['blue', 'green', 'yellow', 'orange', 'red', 'darkred'],
        vmin=hurricane_data['max_wind'].min() * 0.514,
        vmax=hurricane_wind * 0.514,  # Maximum wind speed (hurricane center, convert to m/s)
        caption='Maximum Wind Speed (m/s)'
    )
    wind_colormap.add_to(m)
    
    # Remove custom CSS that might interfere
    
    # Add layer control
    folium.LayerControl(position='bottomright').add_to(m)
    
    # Add title with status information
    is_shutdown = distance_to_farm <= 300  # Assuming 300km is shutdown distance
    title_html = f'''
    <h3 align="center" style="font-size:16px; color:white; background-color:rgba(0,0,0,0.7); padding:10px; margin:10px; border-radius:5px; text-shadow: 2px 2px 4px rgba(0,0,0,0.8);"><b>{title}</b></h3>
    <p align="center" style="font-size:13px; color:white; background-color:rgba(0,0,0,0.7); padding:8px; margin=5px; border-radius:3px; text-shadow: 1px 1px 2px rgba(0,0,0,0.8);">
    Time: {datetime_str} | Distance to Farm: {distance_to_farm:.1f} km | Wind Speed at Farm: {farm_wind_speed_ms:.1f} m/s | 
    Farm Status: <span style="color: {'#ff6b6b' if is_shutdown else '#51cf66'}; font-weight:bold;">
    {"IMPACT ZONE" if is_shutdown else "SAFE ZONE"}</span>
    </p>
    '''
    m.get_root().html.add_child(folium.Element(title_html))
    
    return m

def main():
    """Main function to create the merged map"""
    print("Loading turbine data from SPARQL...")
    turbines_df = load_turbine_data()
    
    print("Loading coordinate data...")
    turbine_locations, windfarm_polygon = load_coordinate_data()
    
    # You would need to load your hurricane data here
    # For now, I'll create a sample closest approach scenario
    print("Creating sample hurricane data for closest approach...")
    
    # Sample data representing Hurricane Sandy's closest approach
    sample_hurricane_data = pd.DataFrame({
        'latitude': [38.8],
        'longitude': [-74.0],
        'max_wind': [75],  # knots
        'datetime': [pd.Timestamp('2012-10-29 21:00')],
        'distance_km': [83.5],
        'min_pressure': [943]
    })
    
    # Create the merged map
    print("Creating merged hurricane-turbine map...")
    merged_map = create_merged_hurricane_turbine_map(
        hurricane_data=sample_hurricane_data,
        turbines_df=turbines_df,
        turbine_locations=turbine_locations,
        windfarm_polygon=windfarm_polygon,
        closest_approach_idx=0,  # Using the first (and only) row
        title='Hurricane Sandy Impact - Detailed Turbine Response'
    )
    
    # Save the map
    output_dir = "/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/Wind/Digital_Twin/Response_hurricane/Output_maps"
    os.makedirs(output_dir, exist_ok=True)
    
    output_file = os.path.join(output_dir, 'merged_hurricane_turbine_detailed_map.html')
    merged_map.save(output_file)
    
    print(f"Merged map saved to: {output_file}")
    print("\nMerged map features:")
    print("- Hurricane center with closest approach details")
    print("- Wind speed impact circles")
    print("- Individual turbines with detailed semantic data")
    print("- Hurricane impact assessment for each turbine")
    print("- Turbine status based on both normal operations and hurricane conditions")
    
    return merged_map

# Execute the main function
if __name__ == "__main__":
    merged_map = main()

Loading turbine data from SPARQL...
Loading coordinate data...
Loaded 121 turbine locations
Loaded wind farm polygon with 157 points
Creating sample hurricane data for closest approach...
Creating merged hurricane-turbine map...
Merged map saved to: /Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/Wind/Digital_Twin/Response_hurricane/Output_maps/merged_hurricane_turbine_detailed_map.html

Merged map features:
- Hurricane center with closest approach details
- Wind speed impact circles
- Individual turbines with detailed semantic data
- Hurricane impact assessment for each turbine
- Turbine status based on both normal operations and hurricane conditions


## Where is the Eyewall?

The **eyewall** is the ring of strongest winds and thunderstorms surrounding the calm eye of a hurricane. It is located at the **radius of maximum wind (RMW)**, denoted as `rmax` in the Holland model. The wind speed reaches its maximum value at the eyewall (`r = rmax`) and decreases both toward the eye and away from the eyewall.